# ResNet50 Architecture Analysis

## Overview

This notebook explores the ResNet50 architecture, which serves as the backbone for DeepLabV3+ segmentation models. Understanding the ResNet50 structure is crucial for modifying and adapting it for semantic segmentation tasks.

### Learning Objectives

1. **Understand ResNet50 Architecture**: Learn the building blocks of ResNet50 including residual blocks, stages, and skip connections
2. **Load Pre-trained Weights**: Load ImageNet pre-trained weights for transfer learning
3. **Visualize Model Structure**: Use Keras's `plot_model` to visualize the architecture
4. **Understand Layer Naming Convention**: Learn how ResNet50 layers are named for easy access and modification

### Key Concepts

- **Residual Blocks**: The fundamental building unit of ResNet50
- **Stages**: Groups of residual blocks with different feature map sizes
- **Skip Connections**: Allow gradients to flow directly through the network
- **Transfer Learning**: Leveraging pre-trained weights for better performance

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, Input, Dense, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model
import warnings
warnings.filterwarnings("ignore")

print(f"TensorFlow version: {tf.__version__}")

## 1. Loading Pre-trained ResNet50

We load ResNet50 with ImageNet weights. The `include_top=False` parameter removes the final classification layers, making it suitable for feature extraction in segmentation tasks.

**Why use pre-trained weights?**
- Faster convergence during training
- Better feature representations learned from large datasets
- Less data required for fine-tuning

In [ ]:
# Load pre-trained ResNet50 without top classification layers
weights_path = './resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5'

# Option 1: Load from local file
try:
    base_model = keras.applications.ResNet50(
        weights=weights_path, 
        include_top=False, 
        input_shape=(256, 256, 3)
    )
    print("Loaded ResNet50 from local weights file.")
except:
    # Option 2: Download from internet
    print("Local weights not found. Downloading from ImageNet...")
    base_model = keras.applications.ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_shape=(256, 256, 3)
    )
    print("ResNet50 loaded successfully from ImageNet.")

## 2. Understanding ResNet50 Architecture

### Layer Naming Convention

ResNet50 uses a specific naming convention for its layers:

```
conv{stage}_block{block_number}_{layer_number}_{type}
```

**Components explained:**
- `stage`: The major processing stage (conv1, conv2, conv3, conv4, conv5)
- `block`: The residual block number within a stage
- `layer_number`: The layer number within a block
- `type`: Layer type (conv, bn, relu, add, etc.)

**Example:** `conv2_block1_1_conv` = Stage 2, Block 1, Layer 1 (convolution layer)

In [ ]:
# Display model summary
base_model.summary()

In [ ]:
# Visualize the model architecture
print("\nGenerating model visualization...")
plot_model(base_model, show_layer_names=True, show_shapes=True, dpi=72)
print("Model visualization saved as 'resnet50_architecture.png'")

## 3. Understanding Key Features of ResNet50

### Feature Map Sizes

ResNet50 progressively reduces spatial dimensions:

| Stage | Input Size | Output Size | Reduction Factor |
|-------|------------|-------------|------------------|
| conv1 | 256x256    | 128x128     | 2x               |
| conv2 | 128x128    | 64x64       | 4x               |
| conv3 | 64x64      | 32x32       | 8x               |
| conv4 | 32x32      | 16x16       | 16x              |
| conv5 | 16x16      | 8x8         | 32x              |

### Residual Block Structure

Each residual block consists of:

```
Input → Conv1x1 → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN → Add → ReLU → Output
       └──────────────────────────────────────────────────────────────────────┘
```

**Key Insight**: The skip connection (shortcut) allows gradients to bypass the convolution layers, enabling training of very deep networks.

In [ ]:
# Explore layer names and their purposes
print("\n=== Layer Name Examples ===")
for layer in base_model.layers[:10]:
    print(f"Layer: {layer.name}, Type: {layer.__class__.__name__}")

print("\n=== Important Layer Names for DeepLabV3+ ===")
important_layers = [
    'conv2_block3_out',   # Low-level features (skip connection)
    'conv4_block6_out',   # High-level features (encoder output)
]
for layer_name in important_layers:
    try:
        layer = base_model.get_layer(layer_name)
        print(f"✓ {layer_name}: Shape {layer.output.shape}")
    except:
        print(f"✗ {layer_name}: Layer not found")

## 4. Modifying ResNet50 for Semantic Segmentation

### Why Modify the Architecture?

For semantic segmentation, we need:
1. **Higher resolution feature maps** (not reduced to 8x8)
2. **Larger receptive field** to capture context
3. **Multi-scale features** for better segmentation

### Modifications for DeepLabV3+

1. **Remove conv5 downsampling**: Use atrous convolution instead of stride-2
2. **Use dilation rates**: Replace stride-2 with dilation rate 2
3. **Extract low-level features**: Use conv2_block3_out for skip connections
4. **Extract high-level features**: Use conv4_block6_out as encoder output

### Impact on Output Stride

| Configuration | Output Stride | Feature Map Size |
|---------------|---------------|------------------|
| Original ResNet50 | 32 | 8x8 |
| Modified (OS=16) | 16 | 16x16 |
| Modified (OS=8) | 8 | 32x32 |

In [ ]:
def analyze_resnet_stages(model):
    """Analyze and display the structure of each stage in ResNet50."""
    
    # Define stage boundaries
    stages = {
        'conv1': {'start': 'conv1_conv', 'end': 'pool1_pool'},
        'conv2': {'start': 'conv2_block1_1_conv', 'end': 'conv2_block3_out'},
        'conv3': {'start': 'conv3_block1_1_conv', 'end': 'conv3_block4_out'},
        'conv4': {'start': 'conv4_block1_1_conv', 'end': 'conv4_block6_out'},
        'conv5': {'start': 'conv5_block1_1_conv', 'end': 'conv5_block3_out'}
    }
    
    print("\n=== ResNet50 Stage Analysis ===")
    print("-" * 60)
    
    for stage_name, boundaries in stages.items():
        try:
            start_layer = model.get_layer(boundaries['start'])
            end_layer = model.get_layer(boundaries['end'])
            
            # Get feature map shape
            start_shape = start_layer.output.shape
            end_shape = end_layer.output.shape
            
            print(f"\n{stage_name.upper()}:")
            print(f"  Start: {start_layer.name} -> Shape: {start_shape}")
            print(f"  End:   {end_layer.name} -> Shape: {end_shape}")
            print(f"  Spatial Reduction: {start_shape[1]} → {end_shape[1]}")
        except Exception as e:
            print(f"\n{stage_name.upper()}: Unable to analyze ({e})")

# Run the analysis
analyze_resnet_stages(base_model)

## 5. Key Takeaways

### Important Points

1. **Layer Access**: Use `model.get_layer(layer_name)` to access specific layers
2. **Feature Extraction**: `conv4_block6_out` and `conv2_block3_out` are key for DeepLabV3+
3. **Naming Convention**: Understanding the naming helps modify the architecture
4. **Transfer Learning**: Pre-trained weights significantly improve performance

### Next Steps

In the next notebook, we'll build the complete DeepLabV3+ model using these insights:
- Implement Atrous Spatial Pyramid Pooling (ASPP)
- Add skip connections for low-level features
- Train the model on brain tumor segmentation